# Concrete Slump & Strength Testing - United Kingdom (BS EN)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedRafiAhmedGIS/civil-engineering-teaching-portfolio/blob/main/code/notebooks/concrete_testing_UK_BSEN.ipynb)

**Region:** United Kingdom / Ireland · **Author:** Syed Rafi Ahmed · **License:** MIT

A complete, self-contained lab session for the concrete slump and
compressive-strength tests. It runs end-to-end on built-in synthetic data;
download the CSV templates in the Data cell and use its Upload button to analyse your own CSVs.

| Item | Value |
|---|---|
| Slump test | BS EN 12350-2 |
| Strength test | BS EN 12390-3 |
| Specimen | 150 mm cube (area 22,500 mm2) |
| Loading rate | 0.6 +/- 0.2 MPa/s |
| Slump classes | S1 10-40, S2 50-90, S3 100-150, S4 160-210, S5 >= 220 mm |
| Acceptance (teaching) | mean of set >= fck + 4.0 MPa (EN 206 initial) |

## 1. Standard profile used in this notebook

The acceptance margin and classification bands below are **teaching
simplifications** of the official specification - see the limitations
section at the end.

In [ ]:
# ---- Setup ----
# Google Colab already provides numpy / pandas / matplotlib.
# The install line is kept so the notebook also works on a fresh local env.
%pip install -q numpy pandas matplotlib


## 2. Data - use the demo, or upload your own

The notebook **runs automatically**: *Runtime > Run all* completes end-to-end on the
built-in synthetic dataset, with no interaction needed.

**To use your own results instead** (three steps):

1. **Get the template** - the next cell creates `upload_template_strength.csv` and
   `upload_template_slump.csv`, shows a preview of each, and in Colab downloads them
   to your computer automatically.
2. **Fill in your measurements** - keep the header row exactly as it is, delete the
   example rows, and add yours (one row per cylinder for strength, one per batch for slump).
3. **Upload** - click the **Upload** button at the bottom of the next cell and pick your
   CSV(s), then re-run the following cell: your data replaces the demo data and a preview
   is shown so you can confirm what was loaded. If you upload nothing, the demo data is kept.

Required columns (spelling matters; extra columns are ignored):

| File | Required columns | Example row |
|---|---|---|
| strength | `batch, age_days, force_kn, area_mm2` | `M30-A, 28, 612.7, 17671.5` |
| slump | `batch, slump_mm, target_class` | `B03, 110, S3` |


In [ ]:
# ---- Data: template download, upload button, and automatic demo fallback ----
import io

import pandas as pd

STRENGTH_COLS = {"batch", "age_days", "force_kn", "area_mm2"}
SLUMP_COLS = {"batch", "slump_mm"}

DEMO_STRENGTH_CSV = """batch,age_days,force_kn,area_mm2
M20-A,7,290.5,22500.0
M20-A,7,297.2,22500.0
M20-A,7,285.8,22500.0
M20-A,28,468.2,22500.0
M20-A,28,477.0,22500.0
M20-A,28,459.7,22500.0
M30-A,7,512.3,22500.0
M30-A,7,523.1,22500.0
M30-A,7,505.8,22500.0
M30-A,28,780.1,22500.0
M30-A,28,792.0,22500.0
M30-A,28,770.4,22500.0
M30-B,7,498.4,22500.0
M30-B,7,515.0,22500.0
M30-B,7,492.7,22500.0
M30-B,28,740.2,22500.0
M30-B,28,756.7,22500.0
M30-B,28,728.5,22500.0
"""
DEMO_SLUMP_CSV = """batch,slump_mm,target_class
B01,45,S2
B02,72,S2
B03,110,S3
B04,38,S1
B05,165,S4
B06,88,S2
B07,205,S4
B08,132,S3
B09,26,S1
B10,228,S5
"""

TEMPLATE_STRENGTH_CSV = '''batch,age_days,force_kn,area_mm2
M20-A,7,230.0,17671.5
M20-A,28,368.0,17671.5
M30-A,28,612.7,17671.5
'''
TEMPLATE_SLUMP_CSV = '''batch,slump_mm,target_class
B01,45,S2
B02,110,S3
B03,165,S4
'''

def _as_strength(frame):
    missing = STRENGTH_COLS - set(frame.columns)
    if missing:
        raise ValueError(f"strength CSV missing columns: {sorted(missing)}")
    frame = frame.copy()
    frame["strength_mpa"] = frame["force_kn"] * 1000.0 / frame["area_mm2"]
    return frame


def _as_slump(frame):
    missing = SLUMP_COLS - set(frame.columns)
    if missing:
        raise ValueError(f"slump CSV missing columns: {sorted(missing)}")
    return frame.copy()


# -- 1) Build the downloadable templates and preview what to upload --
template_strength = pd.read_csv(io.StringIO(TEMPLATE_STRENGTH_CSV))
template_slump = pd.read_csv(io.StringIO(TEMPLATE_SLUMP_CSV))
template_strength.to_csv("upload_template_strength.csv", index=False)
template_slump.to_csv("upload_template_slump.csv", index=False)
print("PREVIEW - upload_template_strength.csv (one row per cylinder):")
print(template_strength.to_string(index=False))
print()
print("PREVIEW - upload_template_slump.csv (one row per batch):")
print(template_slump.to_string(index=False))
print()

# -- 2) In Colab, download the templates automatically --
try:
    from google.colab import files  # type: ignore
    files.download("upload_template_strength.csv")
    files.download("upload_template_slump.csv")
    print("Templates downloaded to your computer (and saved next to the notebook).")
except Exception:
    print("Templates saved next to the notebook: upload_template_strength.csv, upload_template_slump.csv")

# -- 3) Show the upload button (non-blocking: Run all keeps going) --
uploader = None
try:
    import ipywidgets as widgets  # type: ignore
    uploader = widgets.FileUpload(accept=".csv", multiple=True)
    print("\nUpload your filled-in CSV(s) here (or leave empty to keep the demo data):")
    try:
        from IPython.display import display as _display
        _display(uploader)
    except Exception:
        pass
except Exception as exc:
    print(f"\nUpload button unavailable here ({exc}). In Colab you can instead run in a new cell:")
    print("    from google.colab import files; files.upload()")


In [ ]:
# ---- Read the upload (if any), otherwise keep the demo data ----
# After choosing a file with the Upload button above, re-run this cell
# (or Runtime > Run all) to load it.

def _widget_files(value):
    """Normalise ipywidgets 7 (dict) and 8 (tuple of dicts) upload values."""
    if not value:
        return []
    if isinstance(value, dict):
        return [(str(name), bytes(mem)) for name, mem in value.items()]
    return [(str(item.get("name", f"upload_{i}.csv")), bytes(item.get("content", b"")))
            for i, item in enumerate(value)]


user_strength, user_slump = None, None
uploads = _widget_files(uploader.value) if uploader is not None else []
for name, raw in uploads:
    try:
        frame = pd.read_csv(io.BytesIO(raw))
    except Exception as exc:
        print(f"  skipped {name}: not readable as CSV ({exc})")
        continue
    if STRENGTH_COLS <= set(frame.columns):
        user_strength = _as_strength(frame)
        print(f"  strength data loaded from: {name}")
    elif SLUMP_COLS <= set(frame.columns):
        user_slump = _as_slump(frame)
        print(f"  slump data loaded from: {name}")
    else:
        print(f"  skipped {name}: unrecognised columns (need the template columns)")

if user_strength is None and user_slump is None and uploads:
    print("  no usable file found - keeping the demo data.")
if not uploads:
    print("No file uploaded - running on the built-in demo data.")

if user_strength is not None:
    strength_df = user_strength
    print(f"\nYOUR strength data: {len(strength_df)} rows - preview:")
    print(strength_df.head(6).to_string(index=False))
else:
    strength_df = _as_strength(pd.read_csv(io.StringIO(DEMO_STRENGTH_CSV)))

if user_slump is not None:
    slump_df = user_slump
    print(f"\nYOUR slump data: {len(slump_df)} rows - preview:")
    print(slump_df.head(6).to_string(index=False))
else:
    slump_df = _as_slump(pd.read_csv(io.StringIO(DEMO_SLUMP_CSV)))

strength_df.head(6)


## 3. Slump analysis

In [ ]:
# ---- Slump analysis: classify every batch ----
import matplotlib.pyplot as plt

SLUMP_CLASSES = {"S1": (10, 40), "S2": (50, 90), "S3": (100, 150), "S4": (160, 210), "S5": (220, None)}   # label -> (low_mm, high_mm or None)


def slump_classify(slump_mm, classes):
    """EN-style classification; tolerance gaps report as boundary."""
    if not classes:
        return "reported value (no classes in this profile)"
    for label, (low, high) in classes.items():
        if slump_mm >= low and (high is None or slump_mm <= high):
            return label
    lows = sorted(v[0] for v in classes.values())
    if slump_mm < lows[0]:
        return "below range"
    return "boundary"


slump_df["assigned_class"] = [
    slump_classify(v, SLUMP_CLASSES) for v in slump_df["slump_mm"]
]
slump_df["within_target"] = (
    slump_df["assigned_class"] == slump_df["target_class"]
)
print(slump_df.to_string(index=False))
print(f"\nBatches within specified class: "
      f"{slump_df['within_target'].sum()} / {len(slump_df)}")


In [ ]:
# ---- Slump plot ----
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
colors = ["#2f6f8f" if ok else "#c25b34" for ok in slump_df["within_target"]]
ax.bar(slump_df["batch"], slump_df["slump_mm"], color=colors)
ax.axhline(25, color="#8a9aa5", linestyle=":", linewidth=1,
           label="minimum measurable slump (10-25 mm zone)")
ax.set_xlabel("Batch")
ax.set_ylabel("Slump (mm)")
ax.set_title("Slump results - blue = within specified class, red = out")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False)
plt.show()


## 4. Compressive strength analysis

In [ ]:
# ---- Strength statistics per batch and age ----
import re

import numpy as np


def _grade_from_name(batch):
    """Infer a grade target from a batch name such as M20-A, C30/37 or W35."""
    m = re.search(r"(?:^|[^0-9])[MCW](\d{2})(?:$|[^0-9])", str(batch).upper())
    return float(m.group(1)) if m else None


# fck target per batch, inferred from the batch name (M20 -> 20, C30 -> 30, ...).
# A name without a grade falls back to 30.0 MPa - edit FCK_TARGET if your grades differ.
FCK_TARGET = {}
_unparsed = []
for _b in strength_df["batch"].unique():
    _g = _grade_from_name(_b)
    if _g is None:
        _g = 30.0
        _unparsed.append(_b)
    FCK_TARGET[_b] = _g
if _unparsed:
    print(f"Note: no grade found in batch name(s) {_unparsed}; assumed fck = 30.0 MPa "
          f"- edit FCK_TARGET in this cell if your grades differ.")
ACCEPT_MARGIN = 4.0   # teaching margin (MPa) added to fck - see profile

stats = (
    strength_df.groupby(["batch", "age_days"])["strength_mpa"]
    .agg(["mean", "std", "count"])
    .round(2)
)
print(stats)

d28 = strength_df[strength_df["age_days"] == 28]
print("\n28-day acceptance check (per grade - never pool different grades):")
rows = []
for batch, grp in d28.groupby("batch"):
    vals = grp["strength_mpa"].values
    fck_meas = vals.mean() - 1.64 * vals.std(ddof=1)
    required = FCK_TARGET[batch] + ACCEPT_MARGIN
    ok = vals.mean() >= required
    rows.append((batch, vals.mean(), fck_meas, required, "PASS" if ok else "FAIL"))
summary = pd.DataFrame(
    rows, columns=["batch", "mean_28d", "fck_teaching",
                   "required_mean", "acceptance"]
).round(2)
print(summary.to_string(index=False))


In [ ]:
# ---- Strength development plot ----
fig, ax = plt.subplots(figsize=(7, 4.2), constrained_layout=True)
for i, (batch, grp) in enumerate(strength_df.groupby("batch")):
    g = grp.groupby("age_days")["strength_mpa"].agg(["mean", "std"])
    ax.errorbar(g.index, g["mean"], yerr=g["std"].fillna(0), marker="o",
                capsize=4, linewidth=2,
                color=["#2f6f8f", "#c25b34", "#3a7d5c"][i % 3],
                label=batch)
ax.set_xlabel("Age (days)")
ax.set_ylabel("Compressive strength (MPa)")
ax.set_title("Compressive strength development")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.2)
ax.legend(frameon=False)
plt.show()


In [ ]:
# ---- Quality-control chart (28-day batch means vs acceptance line) ----
fig, ax = plt.subplots(figsize=(7, 4.2), constrained_layout=True)
xs = range(len(summary))
bar_colors = ["#2f6f8f" if v == "PASS" else "#c25b34" for v in summary["acceptance"]]
ax.bar(xs, summary["mean_28d"], color=bar_colors)
for x, (req, m) in enumerate(zip(summary["required_mean"], summary["mean_28d"])):
    ax.plot([x - 0.4, x + 0.4], [req, req], color="#22303a", linewidth=1.5)
ax.set_xticks(list(xs))
ax.set_xticklabels(summary["batch"])
ax.set_ylabel("Mean 28-day strength (MPa)")
ax.set_title("Acceptance check - bar vs required line (fck + margin)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()


## Interpretation - fill in as you go

1. Which batches met their specified slump class, and what production factor most
   plausibly explains the ones that missed (water addition on site, mixer timing,
   aggregate moisture)?
2. Compare the 7-day and 28-day means: does the strength ratio look like the
   expected 60-75% early-strength pattern for ordinary Portland cement?
3. For any FAIL batch: is the problem the mean, or the scatter? What would you
   change in production before changing the mix design?


## Self-check quiz

<details><summary><b>Q1.</b> Your slump measured 45 mm in an S2 specification
(50-90 mm). Is the batch acceptable?</summary>
It falls in the tolerance gap below S2 - report it as a boundary result and
check the specification's tolerance rules; most specifications allow acceptance
with documented tolerance, but it is not a clean S2 pass.
</details>

<details><summary><b>Q2.</b> Why is the characteristic strength computed as
mean - 1.64 x standard deviation?</summary>
It estimates the 5th percentile of a normal distribution: 95% of results are
expected to exceed it. Codes add further margins and grouping rules on top of
this teaching estimate.
</details>

<details><summary><b>Q3.</b> Two specimens of the same batch give 33 and 41 MPa.
What do you do before averaging?</summary>
Check for outliers per the standard's rules (faulty preparation, eccentric
loading, curing lapses). Investigate first; discard only with documented cause.
</details>


## References and limitations

- BS EN 12350-2; BS EN 12390-3; EN 206 (slump classes and conformity criteria).
- All datasets used by default are **synthetic**, generated for teaching; no
  real project or student data is included.
- Calculations are **teaching simplifications** (characteristic-strength margin,
  acceptance rules). Always consult the current official standard text for real
  quality-control decisions.
- The full tested toolkit behind this notebook lives in `code/concrete_analysis/`
  in the repository (pytest suite included).

Portfolio author: Syed Rafi Ahmed - civil-engineering-teaching-portfolio (MIT).
